# **Project Name**    - Yes Bank Stock Closing Price Prediction



##### **Project Type**    - EDA & Regression
##### **Contribution**    - Individual

# **Project Summary -**

**Overview:**

Yes Bank has been a prominent banking entity in the Indian financial sector. It experienced noticable stock price volatility over the past two decades. The bank's stock trajectory provides a classic case study of market dynamics, investor sentiment, and corporate governance crises. This project develops a machine learning regression framework to predict the monthly closing price of Yes Bank stock. The dataset spans from the bank's early days in 2005. It explicitly captures the structural collapse triggered by Rana Kapoor (the founder & former CEO) fraud and management crisis around 2018.

**Data Exploration and Characteristics:**

The dataset contains basic stock metrics: Open, High, Low, and Close prices for each month. Initial Exploratory Data Analysis (EDA) revealed a compact dataset of 185 observations. Time-series line plots showed a steady upward growth trajectory peaking at over ₹400 in mid-2018. This peak was followed by a catastrophic downward spiral due to multiple reasons. Distribution plots showed extreme right-skewness across all numerical variables. This violated the normality assumptions essential for stable ordinary least squares (OLS) linear estimation. Statistical tests confirmed this skewness, necessitating target feature conditioning.

**Feature Engineering and Data Preparation:**

An important observation found during EDA was a near-perfect multicollinearity. Pair plots and Variance Inflation Factor (VIF) matrices indicated correlation coefficients of nearly 1.00 among Open, High, and Low prices. This structural redundancy can cause high variance and unstable coefficient estimates in standard linear models. To resolve this without losing historical structural data, the features were transformed using a base-10 logarithmic scale (log10). This converted the skewed distributions into normal distributions. The conditioned features were scaled using a MinMaxScaler to normalize ranges between 0 and 1, ensuring optimal gradient convergence.

**Model Implementation and Methodology:**

The data was split using an 80:20 training-to-testing ratio to evaluate generalization performance. Three distinct regression algorithms were implemented sequentially from which the hypertuned Ridge Regression model (alpha=0.001) was chosen as the best model. Model explainability via SHAP (SHapley Additive exPlanations) confirmed that the monthly floor price (Low_log) acted as the primary baseline anchor for final stock price settlements.

**Deployment Ready Sanity Check:**

The finalized Ridge model and its corresponding MinMaxScaler were serialized using the joblib library. An inference script was successfully run to process unseen raw feature arrays. It wraps inputs into structured pandas DataFrames to prevent feature-name mismatches, applies the log10 scaling pipeline, and reverses the target transformation. This machine learning framework provides a system to forecast closing trends based on monthly open, high, and low indicators.

# **GitHub Link -**

**https://github.com/21f1003070/Innovexis-AIML-Internship/tree/Yes-Bank-Stock-Prediction**

# **Problem Statement**


The historical stock data of Yes Bank, a well-known Bank in the Indian financial domain, contains high volatility. A major reason behind the instability is the 2018 corporate fraud and management crisis involving the founder and former CEO, Rana Kapoor.
The objective is to build a predictive regression model that accurately forecasts the monthly Closing Price of Yes Bank stock.

# **General Guidelines** : -  

1.   Well-structured, formatted, and commented code is required.
2.   Exception Handling, Production Grade Code & Deployment Ready Code will be a plus. Those students will be awarded some additional credits.
     
     The additional credits will have advantages over other students during Star Student selection.
       
             [ Note: - Deployment Ready Code is defined as, the whole .ipynb notebook should be executable in one go
                       without a single error logged. ]

3.   Each and every logic should have proper comments.
4. You may add as many number of charts you want. Make Sure for each and every chart the following format should be answered.
        

```
# Chart visualization code
```
            

*   Why did you pick the specific chart?
*   What is/are the insight(s) found from the chart?
* Will the gained insights help creating a positive business impact?
Are there any insights that lead to negative growth? Justify with specific reason.

5. You have to create at least 15 logical & meaningful charts having important insights.


[ Hints : - Do the Vizualization in  a structured way while following "UBM" Rule.

U - Univariate Analysis,

B - Bivariate Analysis (Numerical - Categorical, Numerical - Numerical, Categorical - Categorical)

M - Multivariate Analysis
 ]





6. You may add more ml algorithms for model creation. Make sure for each and every algorithm, the following format should be answered.


*   Explain the ML Model used and it's performance using Evaluation metric Score Chart.


*   Cross- Validation & Hyperparameter Tuning

*   Have you seen any improvement? Note down the improvement with updates Evaluation metric Score Chart.

*   Explain each evaluation metric's indication towards business and the business impact pf the ML model used.




















# ***Let's Begin !***

## ***1. Know Your Data***

### Import Libraries

In [ ]:
# Import Libraries

# Importing libraries for data manipulation and analysis
import pandas as pd
import numpy as np

# Importing libraries for visualization
import matplotlib.pyplot as plt
import seaborn as sns
%matplotlib inline

# Importing Scikit-learn for model building and evaluation
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression, Lasso, Ridge
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error
from sklearn.preprocessing import MinMaxScaler

# Importing for Statistical analysis
from statsmodels.stats.outliers_influence import variance_inflation_factor

# Set plot style
sns.set_style('whitegrid')


### Dataset Loading

In [ ]:
# Load Dataset

from google.colab import files
uploaded = files.upload()
Yes_Bank = pd.read_csv('data_YesBank_StockPrices.csv')


### Dataset First View

In [ ]:
# Dataset First Look

# Viewing the first 5 rows
Yes_Bank.head()


### Dataset Rows & Columns count

In [ ]:
# Dataset Rows & Columns count

print(f"Rows: {Yes_Bank.shape[0]}, Columns: {Yes_Bank.shape[1]}")


### Dataset Information

In [ ]:
# Dataset Info

Yes_Bank.info()

#### Duplicate Values

In [ ]:
# Dataset Duplicate Value Count

len(Yes_Bank[Yes_Bank.duplicated()])

#### Missing Values/Null Values

In [ ]:
# Missing Values/Null Values Count

Yes_Bank.isnull().sum()


In [ ]:
# Visualizing the missing values

### What did you know about your dataset?

The dataset comprises 185 monthly stock price observations over 5 columns covering July,2005 to November,2020, with zero missing or duplicate values. It is characterized by severe right-skewness and extreme multicollinearity between Open, High, and Low features, requiring transformation and regularization.

## ***2. Understanding Your Variables***

In [ ]:
# Dataset Columns

Yes_Bank.columns

In [ ]:
# Dataset Describe

Yes_Bank.describe(include='all')

### Variables Description

# The dataset consists of 5 variables tracking the monthly Stock Price figures of Yes Bank over 185 observations:

**Date:** A categorical time-series feature representing the month and year of the recorded stock data (e.g. - "Jul-05"). It contains 185 unique entries.

**Open:** A continuous numerical feature representing the starting price of the stock at the beginning of the month. Prices range from a minimum of ₹10.00 to a maximum of ₹369.95, with an average opening price of ₹105.54.

**High:** A continuous numerical feature representing the peak price achieved by the stock during the month. It records the highest historical point in the dataset at ₹404.00, with an overall mean of ₹116.10.

**Low:** A continuous numerical feature representing the lowest floor price reached by the stock in the month. It contains the absolute baseline minimum of the dataset at ₹5.55, with an average floor of ₹94.95.

**Close:** The continuous numerical Target Variable representing the final price at which the stock settled at the end of the month. It ranges from ₹9.98 to ₹367.90, with a monthly average of ₹105.20.

### Check Unique Values for each variable.

In [ ]:
# Check Unique Values for each variable.

Yes_Bank.nunique()

## 3. ***Data Wrangling***

### Data Wrangling Code

In [ ]:
# Write your code to make your dataset analysis ready.

# Converting the 'Date' column to a proper datetime object
Yes_Bank['Date'] = pd.to_datetime(Yes_Bank['Date'], format='%b-%y')

# Sorting the data by date to ensure the time series is in order
Yes_Bank.sort_values('Date', inplace=True)

# Checking for any missing values created during conversion
print(Yes_Bank.isnull().sum())

# Previewing the changes
Yes_Bank.head()


### What all manipulations have you done and insights you found?

# **Data Manipulations Performed:**

**Datetime Parsing:** Converted the Date column from a string object format (e.g. - "Jul-05") into "a standardized pandas datetime object using pd.to_datetime() (e.g - "2005-07-01)

**Chronological Sorting:** Sorted the entire dataset based on the newly formatted Date index in ascending order using .sort_values(inplace=True). This organizes the records sequentially to maintain correct time-series logic.

**Integrity Verification:** Checked for missing or null values across all columns post-conversion using .isnull().sum() to ensure no data corruption or invalid parsing occurred.

# **Insights Discovered:**

**Perfect Data Completeness:** The missing value analysis yielded a count of 0 across all attributes, confirming that the dataset requires no inputation or row-deletion treatments.


**Initial Price Stability:** In its early months (July 2005 - November 2005), the stock exhibits low volatility and highly compressed trading bounds, with monthly opening values tightly hovering between ₹12.58 and ₹13.48, and closing values between ₹12.46 to ₹13.71.

## ***4. Data Vizualization, Storytelling & Experimenting with charts : Understand the relationships between variables***

#### Chart - 1 : Yes Bank Yearly Closing Price Trend

In [ ]:
# Chart - 1 visualization code

plt.figure(figsize=(12, 6))
sns.lineplot(x='Date', y='Close', data=Yes_Bank, color='blue')
plt.title('Yes Bank Yearly Closing Price Trend (2005 - 2020)')
plt.xlabel('Year')
plt.ylabel('Closing Price (INR)')
plt.show()

##### 1. Why did you pick the specific chart?


Its the standard way to visualize stock movement over time. It helps identify trends, cycles, and massive outliers.

##### 2. What is/are the insight(s) found from the chart?

We can see a steady rise until 2018, followed by a drastic crash. This confirms the impact of the Rana Kapoor fraud case.

##### 3. Will the gained insights help creating a positive business impact?
Are there any insights that lead to negative growth? Justify with specific reason.


 Highlighting the 2018 crash leads to "negative growth" insights; it shows that technical indicators alone can't predict "black swan" events like fraud, suggesting a need for risk management models.

#### Chart - 2 : Comparison between skewed original and log-transformed Closing Price Distribution


In [ ]:
# Chart - 2 visualization code

plt.figure(figsize=(12, 6))
plt.subplot(1, 2, 1)
sns.histplot(Yes_Bank['Close'], kde=True, color='purple')
plt.title('Original Distribution of Close Price')

plt.subplot(1, 2, 2)
sns.histplot(np.log10(Yes_Bank['Close']), kde=True, color='green')
plt.title('Log-Transformed Distribution')
plt.show()

##### 1. Why did you pick the specific chart?


To check for skewness. Regression models perform best when data is normally distributed.

##### 2. What is/are the insight(s) found from the chart?


The original data is heavily right-skewed. The log transformation makes it more symmetric (normal), which is better for linear regression model.

##### 3. Will the gained insights help creating a positive business impact?
Are there any insights that lead to negative growth? Justify with specific reason.


Normalizing the data ensures more stable predictions, leading to a positive impact by reducing the model's error rate (RMSE) on high-value stock periods.

#### Chart - 3 : Distribution of Open Prices

In [ ]:
# Chart - 3 visualization code

plt.figure(figsize=(7, 4))
sns.histplot(Yes_Bank['Open'], kde=True, color='blue')
plt.title('Distribution of Open Prices')
plt.show()

##### 1. Why did you pick the specific chart?

To check the shape, spread, and skewness of the Open price feature.

##### 2. What is/are the insight(s) found from the chart?

The feature is highly right-skewed; most trading months opened at lower price ranges.

##### 3. Will the gained insights help creating a positive business impact?
Are there any insights that lead to negative growth? Justify with specific reason.

Indicates a log transformation is required to stabilize linear regression metrics and optimize predictive bounds.

#### Chart - 4 : Distribution of High Prices

In [ ]:
# Chart - 4 visualization code

plt.figure(figsize=(7, 4))
sns.histplot(Yes_Bank['High'], kde=True, color='green')
plt.title('Distribution of High Prices')
plt.show()

##### 1. Why did you pick the specific chart?

To measure the probability density and distribution properties of the High feature.

##### 2. What is/are the insight(s) found from the chart?

Shows heavy right skewness with a long tail stretching toward peak historical values.

##### 3. Will the gained insights help creating a positive business impact?
Are there any insights that lead to negative growth? Justify with specific reason.

Validates that extreme market peaks are rare events.

#### Chart - 5 : Distribution of Low Prices

In [ ]:
# Chart - 5 visualization code

plt.figure(figsize=(7, 4))
sns.histplot(Yes_Bank['Low'], kde=True, color='brown')
plt.title('Distribution of Low Prices')
plt.show()

##### 1. Why did you pick the specific chart?

To inspect the structural behavior and density profile of monthly floor prices.

##### 2. What is/are the insight(s) found from the chart?

Displays a heavy concentration of data points below the ₹100 threshold level.

##### 3. Will the gained insights help creating a positive business impact?
Are there any insights that lead to negative growth? Justify with specific reason.

Alerts risk models that floor thresholds are highly compressed, highlighting structural downside risks.

#### Chart - 6 : Distribution of Close Prices

In [ ]:
# Chart - 6 visualization code

plt.figure(figsize=(7, 4))
sns.histplot(Yes_Bank['Close'], kde=True, color='purple')
plt.title('Distribution of Close Prices')
plt.show()

##### 1. Why did you pick the specific chart?

To evaluate the target variable (close price's) baseline distribution profile before modeling steps.

##### 2. What is/are the insight(s) found from the chart?

Strong right-skewness confirming non-normality in the target price metrics.

##### 3. Will the gained insights help creating a positive business impact?
Are there any insights that lead to negative growth? Justify with specific reason.

It justifies the requirement for log transformation of the target variable.

#### Chart - 7 : Box Plot of Close Price

In [ ]:
# Chart - 7 visualization code

plt.figure(figsize=(6, 3))
sns.boxplot(x=Yes_Bank['Close'], color='red')
plt.title('Box Plot of Target Variable (Close Price)')
plt.show()

##### 1. Why did you pick the specific chart?

To visually isolate statistical outliers and calculate interquartile range spreads.

##### 2. What is/are the insight(s) found from the chart?

Points beyond ₹300 appear as extreme statistical upper outliers.

##### 3. Will the gained insights help creating a positive business impact?
Are there any insights that lead to negative growth? Justify with specific reason.

Proves the 2018 crash created massive anomalies that could destabilize unregularized models.

#### Chart - 8 : Scatter Plot of Open & Close Price

In [ ]:
# Chart - 8 visualization code

plt.figure(figsize=(7, 4))
sns.scatterplot(x='Open', y='Close', data=Yes_Bank, color='darkblue')
plt.title('Scatter Plot: Open vs Close Price')
plt.show()

##### 1. Why did you pick the specific chart?

To determine the nature of the relationship between opening and closing price variables.

##### 2. What is/are the insight(s) found from the chart?

Shows an exceptionally strong, positive linear relationship across all ranges.

##### 3. Will the gained insights help creating a positive business impact?
Are there any insights that lead to negative growth? Justify with specific reason.

Justifies the initial application of linear regression models for prediction purposes.

#### Chart - 9 : Scatter Plot of High & Close Price

In [ ]:
# Chart - 9 visualization code

plt.figure(figsize=(7, 4))
sns.scatterplot(x='High', y='Close', data=Yes_Bank, color='magenta')
plt.title('Scatter Plot: High vs Close Price')
plt.show()

##### 1. Why did you pick the specific chart?

To analyze how maximum monthly ceiling prices vary with the final monthly close valuations.

##### 2. What is/are the insight(s) found from the chart?

Perfect linear alignment with minimal variance or scattering artifacts observed.

##### 3. Will the gained insights help creating a positive business impact?
Are there any insights that lead to negative growth? Justify with specific reason.

Confirms the high descriptive value of peak metrics when computing final asset valuations.

#### Chart - 10 : Scatter Plot of Low & Close Price

In [ ]:
# Chart - 10 visualization code

plt.figure(figsize=(7, 4))
sns.scatterplot(x='Low', y='Close', data=Yes_Bank, color='darkviolet')
plt.title('Scatter Plot: Low vs Close Price')
plt.show()

##### 1. Why did you pick the specific chart?

To analyze relationships between monthly floor values and concluding prices.

##### 2. What is/are the insight(s) found from the chart?


Tightly clustered linear pattern indicating strong mutual dependency between Low & Close Stock prices.

##### 3. Will the gained insights help creating a positive business impact?
Are there any insights that lead to negative growth? Justify with specific reason.

Shows that monthly price floors set rigid boundaries for final stock valuations.

#### Chart - 11 : Yes Bank Yearly Floor Price Trend

In [ ]:
# Chart - 11 visualization code

plt.figure(figsize=(9, 4))
sns.lineplot(x='Date', y='High', data=Yes_Bank, color='black')
plt.title('Historical High Trend Timeline')
plt.show()

##### 1. Why did you pick the specific chart?

A line chart is the standard tool to observe sequential changes in stock price metrics over a long period. Here it maps the historical behavior of the High Price feature to identify historical peaks.

##### 2. What is/are the insight(s) found from the chart?

The stock experienced stable, long-term valuation appreciation from 2005 through 2014, hovering mostly below ₹100. Then a dramatic rise occurred between 2014 and mid-2018when it achieved its lifetime high above ₹400. Then a catastrophic price collapse took place Post-2018.

##### 3. Will the gained insights help creating a positive business impact?
Are there any insights that lead to negative growth? Justify with specific reason.

Are there any insights that lead to negative growth? Justify with specific reason.

#### Chart - 12 : Yes Bank Yearly Opening Price Trend

In [ ]:
# Chart - 12 visualization code

plt.figure(figsize=(9, 4))
sns.lineplot(x='Date', y='Open', data=Yes_Bank, color='green')
plt.title('Historical Open Trend Timeline')
plt.show()

##### 1. Why did you pick the specific chart?

To contrast historical opening trends against the closing prices.

##### 2. What is/are the insight(s) found from the chart?

Opening trends match the closing price peaks seamlessly.

##### 3. Will the gained insights help creating a positive business impact?
Are there any insights that lead to negative growth? Justify with specific reason.

Are there any insights that lead to negative growth? Justify with specific reason.

#### Chart - 13 : Combined OHLC Financial Metrics

In [ ]:
# Chart - 13 visualization code

plt.figure(figsize=(10, 5))
plt.plot(Yes_Bank['Date'], Yes_Bank['Open'], label='Open')
plt.plot(Yes_Bank['Date'], Yes_Bank['High'], label='High')
plt.plot(Yes_Bank['Date'], Yes_Bank['Low'], label='Low')
plt.plot(Yes_Bank['Date'], Yes_Bank['Close'], label='Close', linestyle='--')
plt.legend()
plt.title('Combined OHLC Financial Metrics Overlay')
plt.show()

##### 1. Why did you pick the specific chart?

To evaluate variance, spread, and intersection anomalies among all variables simultaneously.

##### 2. What is/are the insight(s) found from the chart?

Features move completely in lockstep throughout historical growth and contraction phases.

##### 3. Will the gained insights help creating a positive business impact?
Are there any insights that lead to negative growth? Justify with specific reason.

Signals severe multicollinearity risks that require resolution through feature engineering methods.

#### Chart - 14 - Correlation Heatmap

In [ ]:
# Correlation Heatmap visualization code

plt.figure(figsize=(8, 6))
correlation = Yes_Bank.corr()
sns.heatmap(correlation, annot=True, cmap='coolwarm', fmt=".3f")
plt.title('Correlation Between Features')
plt.show()

##### 1. Why did you pick the specific chart?



To identify Multicollinearity. We want to see how strongly Open, High, and Low relate to the Close price.

##### 2. What is/are the insight(s) found from the chart?


You will find near 1.00 correlation between all features. This means the features are redundant and might cause "overfitting" in a standard regression model.

#### Chart - 15 - Pair Plot

In [ ]:
# Pair Plot visualization code

# We exclude 'Date' as it is a time-series index, not a numerical feature for correlation
sns.pairplot(Yes_Bank.drop('Date', axis=1), diag_kind='kde')
plt.suptitle("Pair Plot of Yes Bank Stock Features", y=1.02)
plt.show()

##### 1. Why did you pick the specific chart?


Because it allows us to simultaneously see the distribution of each variable (on the diagonal) and the scatter relationship between all pairs of features. In a stock prediction task, it is the quickest way to visually confirm if the relationship between our independent variables (Open, High, Low Prices) and our target (Close Price) is linear.

##### 2. What is/are the insight(s) found from the chart?

**Extreme Multicollinearity:** All scatter plots show a very tight, straight line. This confirms that Open, High, and Low are almost perfectly correlated with Close Price.

**Skewness:** The KDE (Kernel Density Estimate) plots on the diagonal show that the features are heavily right-skewed, meaning most of the data points are at lower price levels with a few extreme high values.Linearity: The strong linear trend suggests that Linear Regression is a very appropriate starting model for this dataset.

## ***5. Hypothesis Testing***

### Based on your chart experiments, define three hypothetical statements from the dataset. In the next three questions, perform hypothesis testing to obtain final conclusion about the statements through your code and statistical testing.

### Hypothetical Statement - 1

#### 1. State Your research hypothesis as a null hypothesis and alternate hypothesis.



**Null Hypothesis (H0):** There is no significant difference in the mean closing price of Yes Bank stock before and after the September 2018 crisis.

**Alternate Hypothesis (H1):** There is a significant difference in the mean closing price before and after the September 2018 crisis.

#### 2. Perform an appropriate statistical test.

In [ ]:
# Perform Statistical Test to obtain P-Value

from scipy import stats

# Splitting the data based on the crisis period
pre_crisis = Yes_Bank[Yes_Bank['Date'] < '2019-03-01']['Close']
post_crisis = Yes_Bank[Yes_Bank['Date'] >= '2019-03-01']['Close']

# Perform Independent T-Test
t_stat, p_value = stats.ttest_ind(pre_crisis, post_crisis)

print(f"T-statistic: {t_stat}")
print(f"P-value: {p_value}")

# Conclusion logic
alpha = 0.05
if p_value < alpha:
    print("Reject the Null Hypothesis: The difference is statistically significant.")
else:
    print("Fail to reject the Null Hypothesis: No significant difference found.")


##### Which statistical test have you done to obtain P-Value?

I performed a two-sample t-test.

##### Why did you choose the specific statistical test?


I chose this test because it is the standard method for comparing the means of two independent groups (in this case, "pre_crisis" and "post-crisis") to determine if the observed difference in closing prices is due to a specific cause or just random chance.

### Hypothetical Statement - 2

#### 1. State Your research hypothesis as a null hypothesis and alternate hypothesis.


**Null Hypothesis (H0):** There is no linear relationship between the Open Price and the Close Price.

**Alternate Hypothesis (H1):** There is a significant linear relationship between the Open Price and the Close Price.

#### 2. Perform an appropriate statistical test.

In [ ]:
# Perform Statistical Test to obtain P-Value

from scipy.stats import pearsonr

# Calculate Pearson correlation coefficient and P-value
corr, p_value = pearsonr(Yes_Bank['Open'], Yes_Bank['Close'])

print(f"Pearson Correlation Coefficient: {corr}")
print(f"P-value: {p_value}")

if p_value < 0.05:
    print("Reject the Null Hypothesis: There is a significant linear relationship.")
else:
    print("Fail to reject the Null Hypothesis: No significant relationship found.")


##### Which statistical test have you done to obtain P-Value?


I performed the Pearson Correlation Coefficient Test.

##### Why did you choose the specific statistical test?


I chose this test because the Pair Plot showed a very strong linear pattern. Pearson's test is the standard statistical method to measure the strength and direction of a linear relationship between two continuous variables and provides a p-value to ensure the result is statistically significant.

### Hypothetical Statement - 3

#### 1. State Your research hypothesis as a null hypothesis and alternate hypothesis.



**Null Hypothesis (H0):** The Close Price data follows a normal distribution.

**Alternate Hypothesis (H1):** The Close Price data does not follow a normal distribution (it is skewed).

#### 2. Perform an appropriate statistical test.

In [ ]:
# Perform Statistical Test to obtain P-Value

from scipy.stats import shapiro

# Perform Shapiro-Wilk Test for Normality
stat, p_value = shapiro(Yes_Bank['Close'])

print(f"Test Statistic: {stat}")
print(f"P-value: {p_value}")

if p_value < 0.05:
    print("Reject the Null Hypothesis: The data is not normally distributed (Skewed).")
else:
    print("Fail to reject the Null Hypothesis: The data follows a normal distribution.")


##### Which statistical test have you done to obtain P-Value?



I performed the Shapiro-Wilk Test for Normality.

##### Why did you choose the specific statistical test?


I chose this because the Distribution Plot (Histogram) showed heavy right-skewness. The Shapiro-Wilk test is one of the most powerful and widely used tests for normality. Proving that the data is not normally distributed gives justification for the Log Transformation.

## ***6. Feature Engineering & Data Pre-processing***

### 1. Handling Missing Values

In [ ]:
# Handling Missing Values & Missing Value Imputation

# Check for missing values
print(Yes_Bank.isnull().sum())


#### What all missing value imputation techniques have you used and why did you use those techniques?


**Techniques used:** I Checked for nulls using *.isnull().sum()*.

Since the Yes Bank dataset has 0 null values, hence no imputation required.
If there were any, I would have used forward fill for stock data *Yes_Bank.fillna(method='ffill', inplace=True)*

### 2. Handling Outliers

In [ ]:
# Handling Outliers & Outlier treatments

# Visualizing outliers using Boxplot
plt.figure(figsize=(10,5))
sns.boxplot(data=Yes_Bank.drop('Date', axis=1))
plt.title("Checking for Outliers")
plt.show()

# Treatment: Log Transformation
# This handles the extreme values without deleting data
Yes_Bank['Close_log'] = np.log10(Yes_Bank['Close'])
Yes_Bank['Open_log'] = np.log10(Yes_Bank['Open'])
Yes_Bank['High_log'] = np.log10(Yes_Bank['High'])
Yes_Bank['Low_log'] = np.log10(Yes_Bank['Low'])


##### What all outlier treatment techniques have you used and why did you use those techniques?


**Technique used:** Log Transformation.

**Why?:** In the Yes Bank dataset, the 2018 price peak looks like an outlier, but it is actually critical historical data. I used Log Transformation to compress the scale and reduce the impact of extreme values without losing the information those peaks provide.

### 3. Categorical Encoding

In [ ]:
# Encode your categorical columns

# Check for categorical columns
print(Yes_Bank.dtypes)

# The 'Date' column is the only non-numeric column.
# We don't encode it; we extract numerical features if needed.
Yes_Bank['Year'] = Yes_Bank['Date'].dt.year
Yes_Bank['Month'] = Yes_Bank['Date'].dt.month


#### What all categorical encoding techniques have you used & why did you use those techniques?

**Technique used:** Feature Extraction (extracting Year/Month from Date).

**Why?:** There are no categorical variables in this dataset. Encoding is not needed, but breaking the Date into Year and Month allows the regression model to identify seasonal or yearly trends.

### 4. Textual Data Preprocessing
(It's mandatory for textual dataset i.e., NLP, Sentiment Analysis, Text Clustering etc.)

#### 1. Expand Contraction

In [ ]:
# Expand Contraction

#### 2. Lower Casing

In [ ]:
# Lower Casing

#### 3. Removing Punctuations

In [ ]:
# Remove Punctuations

#### 4. Removing URLs & Removing words and digits contain digits.

In [ ]:
# Remove URLs & Remove words and digits contain digits

#### 5. Removing Stopwords & Removing White spaces

In [ ]:
# Remove Stopwords

In [ ]:
# Remove White spaces

#### 6. Rephrase Text

In [ ]:
# Rephrase Text

#### 7. Tokenization

In [ ]:
# Tokenization

#### 8. Text Normalization

In [ ]:
# Normalizing Text (i.e., Stemming, Lemmatization etc.)

##### Which text normalization technique have you used and why?

Answer Here.

#### 9. Part of speech tagging

In [ ]:
# POS Taging

#### 10. Text Vectorization

In [ ]:
# Vectorizing Text

##### Which text vectorization technique have you used and why?

Answer Here.

### 4. Feature Manipulation & Selection

#### 1. Feature Manipulation

In [ ]:
# Manipulate Features to minimize feature correlation and create new features

from statsmodels.stats.outliers_influence import variance_inflation_factor

# Define independent variables (using the log-transformed ones created earlier)
features = ['Open_log', 'High_log', 'Low_log']
X = Yes_Bank[features]

# Calculate VIF for each feature
vif_data = pd.DataFrame()
vif_data["feature"] = X.columns
vif_data["VIF"] = [variance_inflation_factor(X.values, i) for i in range(len(X.columns))]

print(vif_data)


#### 2. Feature Selection

In [ ]:
# Select your features wisely to avoid overfitting

# Feature Selection Strategy based on VIF and SHAP analysis
# High VIF values (>10) prove extreme multicollinearity among Open_log, High_log, and Low_log.

# Since we are using Regression, we aim to handle multicollinearity via regularization.
# Therefore, we keep all three log features to maintain complete historical pricing signals.
selected_features = ['Open_log', 'High_log', 'Low_log']

# Defining the final training and testing matrix variables
X = Yes_Bank[selected_features]
y = Yes_Bank['Close_log']

print("Selected Model Features:", list(X.columns))
print(f"Feature Space Matrix Shape: {X.shape}")


##### What all feature selection methods have you used  and why?

**Methods used:** Correlation Analysis and Variance Inflation Factor (VIF).

**Why?:** To look for Multicollinearity which might make the model unstable.

##### Which all features you found important and why?


Open, High, and Low are all important since they have decent impact in deciding the Closing Price.

### 5. Data Transformation

#### Do you think that your data needs to be transformed? If yes, which transformation have you used. Explain Why?

In [ ]:
# Transform Your data

# Since all numerical independent features (Open, High, Low Prices) are heavily right-skewed
# Applying Base-10 Logarithmic Transformation to handle extreme right skewness
Yes_Bank['Open_log'] = np.log10(Yes_Bank['Open'])
Yes_Bank['High_log'] = np.log10(Yes_Bank['High'])
Yes_Bank['Low_log'] = np.log10(Yes_Bank['Low'])
Yes_Bank['Close_log'] = np.log10(Yes_Bank['Close'])

# Verifying transformation results by checking new skewness values
print("--- Skewness Values Post Log Transformation ---")
for col in ['Open_log', 'High_log', 'Low_log', 'Close_log']:
    print(f"{col}: {Yes_Bank[col].skew():.4f}")



### 6. Data Scaling

In [ ]:
# Scaling your data

from sklearn.preprocessing import MinMaxScaler

# Scaling ensures features are in a similar range (0 to 1)
scaler = MinMaxScaler()
X_scaled = scaler.fit_transform(Yes_Bank[['Open_log', 'High_log', 'Low_log']])


# Which method have you used to scale you data and why?

**Method used:** MinMaxScaler.

**Why?:** Scaling is crucial so that the model doesn't give more weight to features with larger absolute values while learning. It helps the optimization algorithms (like Gradient Descent) converge faster.

### 7. Dimesionality Reduction

##### Do you think that dimensionality reduction is needed? Explain Why?

**Is it needed?:** No.

**Why?:** We have limited number of features. Dimensionality reduction like PCA (Principal Component Analysis) is usually required for datasets with hundreds of features for better results. Here, its better to keep the original features so the model remains interpretable.

In [ ]:
# DImensionality Reduction (If needed)

##### Which dimensionality reduction technique have you used and why? (If dimensionality reduction done on dataset.)

Answer Here.

### 8. Data Splitting

In [ ]:
# Split your data to train and test. Choose Splitting ratio wisely.

# Defining independent and dependent variables
# Using log-transformed features to handle skewness as discussed
X = Yes_Bank[['Open_log', 'High_log', 'Low_log']]
y = Yes_Bank['Close_log']

# Splitting the data
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print(f"Training set size: {X_train.shape[0]}")
print(f"Testing set size: {X_test.shape[0]}")


##### What data splitting ratio have you used and why?


**Splitting Ratio:** 80:20 (80% training, 20% testing).

**Why?:** Since the dataset is relatively small (185 rows), an 80:20 split ensures we keep enough data for the model to learn historical patterns (148 rows) while retaining a sufficient number of unseen samples (37 rows) to get a reliable evaluation of the model's performance.

### 9. Handling Imbalanced Dataset

##### Do you think the dataset is imbalanced? Explain Why.



No. Imbalance is a concept primarily used in Classification (e.g., more "Safe" transactions than "Fraud" transactions). This is a Regression task where we are predicting a continuous price. While the data is skewed, it is not "imbalanced" in the statistical sense that requires resampling techniques like SMOTE.

In [ ]:
# Handling Imbalanced Dataset (If needed)

##### What technique did you use to handle the imbalance dataset and why? (If needed to be balanced)



I did not use imbalance handling techniques because this is a regression problem. Instead, I used Log Transformation in the pre-processing stage to handle the skewness of the target variable, which ensures the model isn't biased toward only the lower price ranges.

## ***7. ML Model Implementation***

### ML Model - 1 : Linear Regression

In [ ]:
# ML Model - 1 Implementation

from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error

# Fitting the Algorithm
reg = LinearRegression()
reg.fit(X_train, y_train)

# Prediction on the model
y_pred_train = reg.predict(X_train)
y_pred_test = reg.predict(X_test)


#### 1. Explain the ML Model used and it's performance using Evaluation metric Score Chart.

In [ ]:
# Visualizing evaluation Metric Score chart

# Function to calculate and print evaluation metrics
def evaluation_metrics(actual, predicted, title):
    mse = mean_squared_error(actual, predicted)
    rmse = np.sqrt(mse)
    r2 = r2_score(actual, predicted)
    print(f"{title} - RMSE: {rmse:.4f}, R2: {r2:.4f}")

evaluation_metrics(y_train, y_pred_train, "Train")
evaluation_metrics(y_test, y_pred_test, "Test")

# Plotting Actual vs Predicted
plt.scatter(y_test, y_pred_test)
plt.xlabel("Actual Log Prices")
plt.ylabel("Predicted Log Prices")
plt.title("Actual vs Predicted (Linear Regression)")
plt.show()


#### 2. Cross- Validation & Hyperparameter Tuning

In [ ]:
# ML Model - 1 Implementation with hyperparameter optimization techniques (i.e., GridSearch CV, RandomSearch CV, Bayesian Optimization etc.)


from sklearn.linear_model import Lasso
from sklearn.model_selection import GridSearchCV

# Initialize Lasso with an increased maximum iteration ceiling
lasso = Lasso(max_iter=10000)

# Define the hyperparameter optimization grid
parameters = {'alpha': [1e-4, 1e-3, 1e-2, 0.1, 1, 5, 10, 20]}

# Setup GridSearchCV with 5-fold Cross-Validation
lasso_regressor = GridSearchCV(lasso, parameters, scoring='neg_mean_squared_error', cv=5)

# Fit the Algorithm
lasso_regressor.fit(X_train, y_train)
best_lasso = lasso_regressor.best_estimator_

# Predict on the model
y_pred_lasso = best_lasso.predict(X_test)


##### Which hyperparameter optimization technique have you used and why?


**Technique:** GridSearchCV.

**Why?:** It systematically works through multiple combinations of the Alpha parameter (regularization strength) to find the optimal value that prevents overfitting while maintaining accuracy.

##### Have you seen any improvement? Note down the improvement with updates Evaluation metric Score Chart.

In [ ]:
# Calculating Lasso Metrics on the Log Scale
mse_lasso = mean_squared_error(y_test, y_pred_lasso)
rmse_lasso = np.sqrt(mse_lasso)
r2_lasso = r2_score(y_test, y_pred_lasso)

# Compiling the results into a comparative DataFrame
summary_metrics = pd.DataFrame({
    'Model type': ['Linear Regression (Baseline)', 'Lasso Regression (Hypertuned)'],
    'Test RMSE': [0.0416, round(rmse_lasso, 4)],
    'Test R2 Score': [0.9888, round(r2_lasso, 4)]
})

print("--- Updated Evaluation Metric Score Chart ---")
display(summary_metrics)




Yes. The R2 score increases slightly and the Generalization Error (difference between Train and Test RMSE) decreases a bit. Lasso helps by shrinking the coefficients of redundant features, making the model more stable during the high-volatility periods like 2018 or onwards.

### ML Model - 2 : Ridge Regression

In [ ]:
from sklearn.linear_model import Ridge

# 1. Fit the Algorithm
ridge = Ridge()
ridge.fit(X_train, y_train)

# 2. Predict on the model
y_pred_ridge_train = ridge.predict(X_train)
y_pred_ridge_test = ridge.predict(X_test)


#### 1. Explain the ML Model used and it's performance using Evaluation metric Score Chart.

In [ ]:
# Visualizing evaluation Metric Score chart

evaluation_metrics(y_train, y_pred_ridge_train, "Ridge Train Set")
evaluation_metrics(y_test, y_pred_ridge_test, "Ridge Test Set")

# Comparing & plotting Actual vs Ridge Predicted
plt.figure(figsize=(10,5))
plt.plot(10**y_test.values, label="Actual") # Converting back from log for real price view
plt.plot(10**y_pred_ridge_test, label="Ridge Predicted")
plt.title("Actual vs Predicted Price (Ridge Regression)")
plt.legend()
plt.show()


#### 2. Cross- Validation & Hyperparameter Tuning

In [ ]:
# ML Model - 1 Implementation with hyperparameter optimization techniques (i.e., GridSearch CV, RandomSearch CV, Bayesian Optimization etc.)

# Ridge Implementation with GridSearchCV
ridge_params = {'alpha': [0.001, 0.01, 0.1, 1, 10, 100, 1000]}
ridge_cv = GridSearchCV(Ridge(), ridge_params, scoring='neg_mean_squared_error', cv=5)

# Fit the Algorithm
ridge_cv.fit(X_train, y_train)
print(f"Best Alpha for Ridge: {ridge_cv.best_params_}")

# Predict on the model
best_ridge = ridge_cv.best_estimator_
y_pred_tuned_ridge = best_ridge.predict(X_test)


##### Which hyperparameter optimization technique have you used and why?


**Technique:** GridSearchCV with 5-fold Cross-Validation.

**Why?:** By testing across 5 different folds, we find a parameter that makes the model robust across the entire history of Yes Bank's stock price movement.

##### Have you seen any improvement? Note down the improvement with updates Evaluation metric Score Chart.

In [ ]:
# Calculating the Hypertuned Ridge Metrics on the Log Scale
mse_tuned_ridge = mean_squared_error(y_test, y_pred_tuned_ridge)
rmse_tuned_ridge = np.sqrt(mse_tuned_ridge)
r2_tuned_ridge = r2_score(y_test, y_pred_tuned_ridge)

# Compiling the results into a comparative DataFrame
summary_ridge_metrics = pd.DataFrame({
    'Model Type': ['Ridge Regression (Baseline)', 'Ridge Regression (Hypertuned, alpha=0.001)'],
    'Test RMSE': [0.0481, round(rmse_tuned_ridge, 4)],
    'Test R2 Score': [0.9851, round(r2_tuned_ridge, 4)]
})

print("--- Updated Ridge Evaluation Metric Score Chart ---")
display(summary_ridge_metrics)



Yes. The Test RMSE decreased from 0.0481 to 0.0416, showing a clear reduction in average prediction error whereas the Test R2 Score improved from 0.9851 to 0.9888, confirming a more stable fit on the log-scaled test data. Setting alpha=0.001 relaxed the L2 penalty just enough to handle collinearity without triggering underfitting behavior.

### ML Model - 3 : XGBoost

In [ ]:
# ML Model - 3 Implementation

from xgboost import XGBRegressor

# Fit the Algorithm
xgb_model = XGBRegressor(n_estimators=100, learning_rate=0.1, max_depth=5, random_state=42)
xgb_model.fit(X_train, y_train)

# Predict on the model
y_pred_xgb_train = xgb_model.predict(X_train)
y_pred_xgb_test = xgb_model.predict(X_test)


#### 1. Explain the ML Model used and it's performance using Evaluation metric Score Chart.

In [ ]:
# Visualizing evaluation Metric Score chart

# Checking metrics
evaluation_metrics(y_train, y_pred_xgb_train, "XGBoost Train Set")
evaluation_metrics(y_test, y_pred_xgb_test, "XGBoost Test Set")

# Plot
plt.figure(figsize=(10,5))
sns.residplot(x=y_test, y=y_pred_xgb_test, color="orange")
plt.title("Plot - XGBoost")
plt.show()


#### 2. Cross- Validation & Hyperparameter Tuning

In [ ]:
# ML Model - 3 Implementation with hyperparameter optimization techniques (i.e., GridSearch CV, RandomSearch CV, Bayesian Optimization etc.)

# Hyperparameter Tuning for XGBoost
xgb_params = {
    'n_estimators': [50, 100, 150],
    'max_depth': [3, 5, 7],
    'learning_rate': [0.01, 0.1, 0.2]
}

xgb_cv = GridSearchCV(XGBRegressor(), xgb_params, scoring='neg_mean_squared_error', cv=3)

# Fit the Algorithm
xgb_cv.fit(X_train, y_train)
best_xgb = xgb_cv.best_estimator_

# Predict on the model
y_pred_tuned_xgb = best_xgb.predict(X_test)


##### Which hyperparameter optimization technique have you used and why?


**Technique:** GridSearchCV.

**Why?:** XGBoost has many "knobs" to turn (like max_depth and learning_rate). GridSearchCV helps us find the sweet spot where the model is complex enough to capture the trends but simple enough to avoid overfitting to the training data.

##### Have you seen any improvement? Note down the improvement with updates Evaluation metric Score Chart.

In [ ]:
# Calculating Hypertuned XGBoost Metrics on the Log Scale
mse_tuned_xgb = mean_squared_error(y_test, y_pred_tuned_xgb)
rmse_tuned_xgb = np.sqrt(mse_tuned_xgb)
r2_tuned_xgb = r2_score(y_test, y_pred_tuned_xgb)

# Compiling the results into a comparative DataFrame
summary_xgb_metrics = pd.DataFrame({
    'Model Type': ['XGBoost (Baseline)', 'XGBoost (Hypertuned via GridSearch)'],
    'Test RMSE': [0.0787, round(rmse_tuned_xgb, 4)],
    'Test R2 Score': [0.9600, round(r2_tuned_xgb, 4)]
})

print("--- Updated XGBoost Evaluation Metric Score Chart ---")
display(summary_xgb_metrics)


No, the hypertuned XGBoost model did not show an improvement from the baseline model. The Test RMSE increased from 0.0787 to 0.0854, and the Test R2 Score decreased from 0.9600 to 0.9529. For this specific case, the default XGBoost hyperparameters managed to capture the stock's historical volatility bounds more effectively than the hypertuned constraints.

### 1. Which Evaluation metrics did you consider for a positive business impact and why?

**Root Mean Squared Error (RMSE):** Selected as the primary metric because it heavily penalizes larger errors by squaring deviations. In stock valuation, large mispredictions can be fatal for risk management purposes.

**R2 Score:** Used to measure the proportion of variance in the closing price explained by the model features. A high R2 confirms that the underlying structural trends of the stock are captured properly.

### 2. Which ML model did you choose from the above created models as your final prediction model and why?

**Selected Model:** Hypertuned Ridge Regression (alpha=0.001).

**Why?:** While advanced models like XGBoost can overfit small financial datasets, Ridge uses L2 regularization to distribute coefficient weights safely across collinear features (Open, High, Low Prices).It also lowered the test RMSE down to 0.0416 and stabilizing the test R2 score at 0.9888 not being deviated by extreme outlier events like the 2018 disaster, making it the safest option for real-world application.

### 3. Explain the model which you have used and the feature importance using any model explainability tool?

In [ ]:
# Installing SHAP library
!pip install shap -q

import shap
import matplotlib.pyplot as plt

# Initializing the SHAP Explainer using final selected model and training data
# We use the training set as a baseline to compute feature contributions
explainer = shap.LinearExplainer(best_ridge, X_train)

# Calculating SHAP values for the test dataset
shap_values = explainer(X_test)

# Generating the SHAP Summary Plot to show Feature Importance and Impact Direction
plt.figure(figsize=(8, 5))
plt.title("SHAP Feature Importance & Impact Direction (Yes Bank Close Price)", fontsize=12, fontweight='bold', pad=20)

# The summary plot ranks features by importance and uses color to show feature value (High = Red, Low = Blue)
shap.summary_plot(shap_values, X_test, feature_names=X_test.columns, plot_type="bar", show=False)

plt.xlabel("Mean Absolute SHAP Value (Impact Magnitude)")
plt.tight_layout()
plt.show()


**Model Explanation:** The selected model is a regularized linear framework operating on log-transformed stock price values. It minimizes the residual sum of squares while adding a penalty proportional to the square of the coefficient magnitudes to prevent weights from exploding due to feature correlation.

**Explainability Tool Used:** SHAP (SHapley Additive exPlanations).

**Feature Importance Ranking:**

Primary Driver (Low_log): It exhibits the highest mean absolute SHAP value (~0.26), confirming that the monthly floor price functions as the strongest mathematical baseline anchor for predicting the final monthly price.

Secondary Driver (High_log): It holds the second highest impact magnitude (~0.25), reflecting market momentum ceilings and upper-bound investor sentiment over the monthly trading cycle.

Marginal Driver (Open_log): It records the lowest SHAP impact magnitude (~0.19), validating that early-month open prices carry less predictive weight as intra-month structural volatility might shift the final price.

## ***8.*** ***Future Work (Optional)***

### 1. Save the best performing ml model in a pickle file or joblib file format for deployment process.


In [ ]:
# Save the File

import joblib

best_model = best_ridge

# Saving the best model and preprocessing MinMaxScaler artifact
joblib.dump(best_model, 'yes_bank_stock_predictor.joblib')
joblib.dump(scaler, 'features_scaler.joblib')

print("Model and Scaler successfully saved for deployment!")

print("Files generated: 'yes_bank_stock_predictor.joblib', 'features_scaler.joblib'")

### 2. Again Load the saved model file and try to predict unseen data for a sanity check.


In [ ]:
# Load the File and predict unseen data.

import warnings

# Suppressing pandas feature-name tracking layout alerts during inference
warnings.filterwarnings('ignore', category=UserWarning)

# Reloading the serialized model and scaler artifacts from disk storage
loaded_model = joblib.load('yes_bank_stock_predictor.joblib')
loaded_scaler = joblib.load('features_scaler.joblib')

# Defining random production inputs simulating a new month
unseen_input = np.array([[35.5, 38.2, 33.1]])

# Applying base-10 log transformation to align with model preprocessing architecture
unseen_log = np.log10(unseen_input)

# Wrapping transformed input into a structured DataFrame to maintain feature name integrity
feature_cols = ['Open_log', 'High_log', 'Low_log']
unseen_Yes_Bank = pd.DataFrame(unseen_log, columns=feature_cols)

# Applying the fitted scaler bounds and generate log-scale prediction
unseen_scaled = loaded_scaler.transform(unseen_Yes_Bank)
predicted_log_close = loaded_model.predict(unseen_scaled)

# Applying inverse log transformation (10^x) to extract the actual target price valuation in INR
final_predicted_price = 10 ** predicted_log_close[0]

print("--- Sanity Check Execution Results ---")
print(f"Raw Input Features Passed -> Open: ₹35.5, High: ₹38.2, Low: ₹33.1")
print(f"Production Sanity Check: Successful")
print(f"Predicted Monthly Closing Stock Price: ₹{final_predicted_price:.2f}")


### ***Congrats! Your model is successfully created and ready for deployment on a live server for a real user interaction !!!***

# **Conclusion**

**Identified Feature Redundancy:** Exploratory analysis confirmed that monthly Open, High, and Low prices are highly collinear variables with correlation coefficients even reaching above 0.98.

**Addressed Skewness Variance:** Plotting distribution revealed heavy right-skewness across features, which was successfully normalized using a log-transformation strategy before model execution.

**Validated Regularization Value:** Regularized linear models (Ridge and Lasso) matched baseline R2 scores while minimizing overall generalization errors during periods of extreme price drops.

**Delivered Production Pipeline:** The project concluded with a serialized inference routine that is extremely helpful for forecasting purposes.

### ***Hurrah! You have successfully completed your Machine Learning Capstone Project !!!***